In [48]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [4]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [7]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [8]:
text[:100]

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'

In [26]:
chars = sorted(list(set(text)))
print(chars)
len(chars)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


65

In [27]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s:str):
    out = [stoi[ch] for ch in s]
    return out

def decode(l):
    out = "".join(itos[i] for i in l)
    return out


In [30]:
data = torch.tensor(encode(text), dtype=torch.long)

In [31]:
n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

In [33]:
train_data.shape

torch.Size([1003854])

In [35]:
val_data.shape

torch.Size([111540])

In [42]:
block_size = 8
batch_size = 4

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size , (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y 
    

In [47]:
x,y = get_batch("train")
print(x,y)

tensor([[43, 57, 43,  1, 42, 39, 63, 57],
        [43,  6,  1, 47, 44,  1, 47, 58],
        [58, 46, 47, 58, 46, 43, 56, 11],
        [33, 32, 27, 24, 37, 15, 33, 31]]) tensor([[57, 43,  1, 42, 39, 63, 57, 11],
        [ 6,  1, 47, 44,  1, 47, 58,  1],
        [46, 47, 58, 46, 43, 56, 11,  1],
        [32, 27, 24, 37, 15, 33, 31, 10]])


In [120]:
vocab_size = 65
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forwards(self,idx,target = None):
        
        logits = self.token_embedding_table(idx) #(B,T,C)
        if target == None:
            loss = None
        else:  
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            target = target.view(B*T)
            loss = nn.functional.cross_entropy(logits, target)

        return logits,loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits,loss = self.forwards(idx)  #(B,T,C)
            logits = logits[:,-1,:]  #(B,C)
            probs = F.softmax(logits,dim = -1)  #(B,C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx,idx_next),dim =1)
        return idx

In [121]:
m = BigramLanguageModel(vocab_size)
logits,loss = m.forwards(x,y)
print(loss.item())

4.867255687713623


In [122]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [133]:
for i in range(10000):
    xb,yb = get_batch("train")
    logits,loss = m.forwards(x,y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(loss.item())

0.4875050187110901


In [135]:
context = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))


3ZRJGTOLYC:Zp3Rf if if dayse, ithe, ithes; if ithe, it ither; days; ithe days; it days; ithithe dayse, ithe days; !YCUTOLYCUS:OLYCUTOLYCUS:DWLYCUTOLYCUTOLYCUS:dayses; ithif dayse it it ither; days; dayser; it days; it if ifUS:zckn?$I, days; ithit days; ithithit days; days; ithe if it ithits; days; days; dayse if if if it if dayse ithithes; dayse, if if days; ithif dayser; days; , dayser; daysese days; ither; it dayse it if days; dayses; days; ithithif days; if days; ithithit ither; ithit it days
